# Custom High-Scale Wake Word Training Pipeline

**Target phrase**: Hello DJ  
**Model name**: Hello_DJ (auto-derived from target_phrase spaces).  
**Scale**: Matched to hey_jarvis production config - 200k positive samples, 50k training steps.

In [1]:
# Step 1: Download MIT RIRs and setup background audio datasets
import os, shutil
from huggingface_hub import hf_hub_download, list_repo_files
from tqdm import tqdm

repo_id = "davidscripka/MIT_environmental_impulse_responses"
output_dir = "/home/jovyan/mit_rirs/16khz"
os.makedirs(output_dir, exist_ok=True)

print("Downloading MIT environmental impulse responses...")
rir_files = [f for f in list_repo_files(repo_id, repo_type='dataset')
             if f.startswith('16khz/') and f.endswith('.wav')]
for fname in tqdm(rir_files, desc="MIT RIRs"):
    hf_hub_download(repo_id=repo_id, filename=fname, repo_type='dataset', local_dir="/home/jovyan/mit_rirs")

n_rirs = len([f for f in os.listdir(output_dir) if f.endswith('.wav')])
print(f"mit_rirs: {n_rirs} wav files in 16khz/")

os.makedirs('/tmp/audioset_16k', exist_ok=True)
src_pos = '/home/jovyan/open-wakeword-repo/notebooks/training_tutorial_data/positive'
if os.path.isdir(src_pos):
    for w in [f for f in os.listdir(src_pos) if f.endswith('.wav')]:
        shutil.copy(os.path.join(src_pos, w), '/tmp/audioset_16k/')
print(f'audioset_16k: {len(os.listdir("/tmp/audioset_16k"))} wav files')

impulse_src = '/home/jovyan/piper-sample-generator/impulses'
if os.path.isdir(impulse_src):
    for w in os.listdir(impulse_src):
        if w.endswith('.wav'):
            shutil.copy(os.path.join(impulse_src, w), output_dir)
print(f'mit_rirs after impulses copy: {len([f for f in os.listdir(output_dir) if f.endswith(".wav")])} wav files')

In [2]:
# Step 2: Download pre-computed openWakeWord features
import os
from huggingface_hub import hf_hub_download

HF_TOKEN = None
print("Downloading pre-computed openWakeWord features...")

hf_hub_download(repo_id="davidscripka/openwakeword_features", filename="validation_set_features.npy",
                repo_type="dataset", token=HF_TOKEN, local_dir="/home/jovyan")
hf_hub_download(repo_id="davidscripka/openwakeword_features", filename="openwakeword_features_ACAV100M_2000_hrs_16bit.npy",
                repo_type="dataset", token=HF_TOKEN, local_dir="/home/jovyan")

print("Pre-computed features downloaded:")
val_path = "/home/jovyan/validation_set_features.npy"
feat_path = "/home/jovyan/openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
print(f"  validation_set_features.npy: {os.path.getsize(val_path) / 1e6:.1f} MB")
print(f"  openwakeword_features_ACAV100M_2000_hrs_16bit.npy: {os.path.getsize(feat_path) / 1e6:.1f} MB")

## Step 3: Setup openwakeword and dependencies

In [ ]:
# Step 3a: Install openwakeword, patch piper, and download model config
import sys, os, subprocess

# Clone openwakeword source if not present
oww_dir = "/home/jovyan/openwakeword"
if not os.path.isdir(oww_dir):
    print("Cloning openWakeWord...")
    subprocess.run(["git", "clone", "--depth=1",
        "https://github.com/dscripka/openWakeWord.git", oww_dir], check=True)
else:
    print("openWakeWord source already present")

sys.path.insert(0, oww_dir)

try:
    import openwakeword
    print(f"openwakeword already installed at {openwakeword.__file__}")
except (ImportError, AttributeError):
    get_ipython().system("pip install -e /home/jovyan/openwakeword 2>&1 | tail -5")

get_ipython().system("pip install --user --break-system-packages webrtcvad espeak-phonemizer 2>&1 | tail -3")
get_ipython().system("pip install 'numpy<2' piper-tts 2>&1 | tail -3")
get_ipython().system("pip install --user --break-system-packages onnxscript tensorflow 2>&1 | tail -3")
get_ipython().system("pip install --user --break-system-packages 'scipy<1.17' 2>&1 | tail -3")
get_ipython().system("pip install --user --break-system-packages onnxruntime-gpu 2>&1 | tail -3")
print("Dependencies OK")

# Clone piper-sample-generator if not present
psg_dir = "/home/jovyan/piper-sample-generator"
if not os.path.isdir(psg_dir):
    print("Cloning piper-sample-generator...")
    subprocess.run(["git", "clone", "--depth=1",
        "https://github.com/dscripka/piper-sample-generator.git", psg_dir], check=True)
else:
    print("piper-sample-generator already present")

# Patch generate_samples.py for weights_only=False
gen_path = os.path.join(psg_dir, "generate_samples.py")
with open(gen_path, 'r') as f:
    content = f.read()
content = content.replace('torch.load(model_path)', 'torch.load(model_path, weights_only=False)')
with open(gen_path, 'w') as f:
    f.write(content)
print("Patched generate_samples.py for weights_only=False")

# Download model JSON config if missing
import requests
models_dir = os.path.join(psg_dir, "models")
json_path = os.path.join(models_dir, "en-us-libritts-high.pt.json")
if not os.path.exists(json_path) or os.path.getsize(json_path) == 0:
    print("Downloading model JSON config...")
    url = 'https://raw.githubusercontent.com/rhasspy/piper-sample-generator/v2.0.0/models/en-us-libritts-high.pt.json'
    r = requests.get(url)
    os.makedirs(models_dir, exist_ok=True)
    with open(json_path, 'wb') as f:
        f.write(r.content)
    print(f"Downloaded {len(r.content)} bytes")
else:
    print(f"Model JSON config exists ({os.path.getsize(json_path)} bytes)")

# Download LibriTTS model .pt file if missing
model_pt = os.path.join(models_dir, "en-us-libritts-high.pt")
if not os.path.exists(model_pt):
    print("Downloading LibriTTS model (~1.3 GB)...")
    url = 'https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt'
    get_ipython().system("wget -O '" + model_pt + "' '" + url + "' 2>&1 | tail -5")
    print(f"Downloaded model ({os.path.getsize(model_pt) / 1e9:.1f} GB)")
else:
    print(f"Model already exists ({os.path.getsize(model_pt) / 1e9:.1f} GB)")

# Fix data.py to use exist_ok
import_path = os.path.join(oww_dir, "openwakeword", "data.py")
with open(import_path, 'r') as f:
    content = f.read()
content = content.replace(
    'os.mkdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), "resources"))',
    'os.makedirs(os.path.join(os.path.dirname(os.path.abspath(__file__)), "resources"), exist_ok=True)')
with open(import_path, 'w') as f:
    f.write(content)
print("Patched data.py with exist_ok=True")

# Patch data.py for stereo RIRs + resampling
with open(import_path, 'r') as f:
    content = f.read()
old = 'rir_waveform, sr = torchaudio.load(random.choice(RIR_paths))\n            augmented_batch = reverberate(augmented_batch.cpu(), rir_waveform, rescale_amp="avg")'
new = 'rir_waveform, sr = torchaudio.load(random.choice(RIR_paths))\n            # Convert stereo/multi-channel to mono\n            if rir_waveform.shape[0] > 1:\n                rir_waveform = rir_waveform.mean(dim=0, keepdim=True)\n            # Resample to 16kHz if needed\n            if sr != 16000:\n                rir_waveform = torchaudio.functional.resample(rir_waveform, sr, 16000)\n                sr = 16000\n            augmented_batch = reverberate(augmented_batch.cpu(), rir_waveform, rescale_amp="avg")'
if old in content:
    content = content.replace(old, new)
    with open(import_path, 'w') as f:
        f.write(content)
    print("Patched data.py for stereo RIRs + resampling")
else:
    print("data.py RIR patch already applied")

# Patch train.py opset_version, dynamo=False, and use custom TFLite converter
train_path = os.path.join(oww_dir, "openwakeword", "train.py")
with open(train_path, 'r') as f:
    content = f.read()
content = content.replace('opset_version=13', 'opset_version=17')
content = content.replace('opset_version=17)', 'opset_version=17, dynamo=False)')
# Add n_blocks support
content = content.replace(
    'layer_dim=config["layer_size"], seconds_per_example=1280*input_shape[0]/16000)',
    'layer_dim=config["layer_size"], n_blocks=config.get("n_blocks", 1), seconds_per_example=1280*input_shape[0]/16000)')
# Use custom TFLite converter instead of onnx_tf
old_tflite = '''        # Convert the model from onnx to tflite format
        if args.convert_to_tflite:
            convert_onnx_to_tflite(os.path.join(config["output_dir"], config["model_name"] + ".onnx"),
                                   os.path.join(config["output_dir"], config["model_name"] + ".tflite"))'''
new_tflite = '''        # Convert the model from onnx to tflite format
        if args.convert_to_tflite:
            try:
                sys.path.insert(0, '/home/jovyan/hellodj/training')
                from convert_onnx_to_tflite import convert as tflite_convert
                tflite_convert(os.path.join(config["output_dir"], config["model_name"] + ".onnx"),
                               os.path.join(config["output_dir"], config["model_name"] + ".tflite"))
            except Exception as e:
                logging.warning(f"Skipping TFLite conversion: {e}")'''
if old_tflite in content:
    content = content.replace(old_tflite, new_tflite)
    print("Patched train.py TFLite conversion with custom converter")
else:
    print("train.py TFLite patch already applied or format differs")
with open(train_path, 'w') as f:
    f.write(content)
print("Patched train.py opset_version, dynamo=False, n_blocks, and TFLite conversion")

# Copy generate_samples.py and piper_train into openwakeword
get_ipython().system("cp /home/jovyan/piper-sample-generator/generate_samples.py /home/jovyan/openwakeword/openwakeword/generate_samples.py")
get_ipython().system("cp -r /home/jovyan/piper-sample-generator/piper_train /home/jovyan/openwakeword/openwakeword/piper_train")
print("Copied generate_samples.py and piper_train module")

# Download feature models (melspectrogram + embedding) for openWakeWord
res_dir = os.path.join(oww_dir, "openwakeword", "resources", "models")
os.makedirs(res_dir, exist_ok=True)
feature_urls = [
    "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx",
    "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite",
    "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx",
    "https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite",
]
for url in feature_urls:
    fname = url.split("/")[-1]
    path = os.path.join(res_dir, fname)
    if not os.path.exists(path):
        print(f"Downloading {fname}...")
        get_ipython().system("wget -O '" + path + "' '" + url + "' 2>&1 | tail -3")
        print(f"  {os.path.getsize(path) / 1e6:.1f} MB")
    else:
        print(f"{fname} already exists ({os.path.getsize(path) / 1e6:.1f} MB)")
print("Feature models ready")

# Ensure my_custom_model.yml exists with all required keys
import yaml

yml_path = "/home/jovyan/my_custom_model.yml"
required_defaults = {
    'model_name': 'Hello_DJ',
    'target_phrase': ['Hello DJ'],
    'n_samples': 200000,
    'n_samples_val': 5000,
    'steps': 50000,
    'model_type': 'dnn',
    'layer_size': 64,
    'n_blocks': 1,
    'augmentation_rounds': 2,
    'tts_batch_size': 8,
    'augmentation_batch_size': 16,
    'piper_sample_generator_path': '/home/jovyan/piper-sample-generator',
    'mit_rir_dir': '/home/jovyan/mit_rirs/16khz',
    'rir_paths': ['/home/jovyan/mit_rirs/16khz'],
    'background_paths': ['/tmp/audioset_16k'],
    'background_paths_duplication_rate': [1],
    'false_positive_validation_data_path': '/home/jovyan/validation_set_features.npy',
    'output_dir': '/home/jovyan/Hello_DJ',
    'target_false_positives_per_hour': 0.2,
    'max_negative_weight': 1500,
    'feature_data_files': {
        'ACAV100M_sample': '/home/jovyan/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
    },
    'batch_n_per_class': {
        'ACAV100M_sample': 1024,
        'adversarial_negative': 50,
        'positive': 50,
    },
    'positive_train_output_path': '',
    'positive_test_output_path': '',
    'negative_train_output_path': '',
    'negative_test_output_path': '',
    'custom_negative_phrases': [],
}

if os.path.exists(yml_path):
    with open(yml_path) as f:
        config = yaml.safe_load(f.read())
    for k, v in required_defaults.items():
        if k not in config:
            config[k] = v
    config['tts_batch_size'] = 8
    with open(yml_path, 'w') as f:
        yaml.dump(config, f)
    print(f"Config file updated with any missing keys at {yml_path}")
else:
    with open(yml_path, 'w') as f:
        yaml.dump(required_defaults, f)
    print(f"Generated default config at {yml_path}")

In [ ]:
# Step 3b: Read, update, and save training config
import yaml, os

yml_path = '/home/jovyan/my_custom_model.yml'
if not os.path.exists(yml_path):
    print(f"Config file not found at {yml_path}, generating defaults")
    config = {
        'model_name': 'Hello_DJ',
        'target_phrase': ['Hello DJ'],
        'n_samples': 200000,
        'steps': 50000,
        'model_type': 'dnn',
        'layer_size': 64,
        'augmentation_rounds': 2,
        'tts_batch_size': 32,
        'piper_sample_generator_path': '/home/jovyan/piper-sample-generator',
        'mit_rir_dir': '/home/jovyan/mit_rirs/16khz',
        'rir_paths': ['/home/jovyan/mit_rirs/16khz'],
        'background_paths': ['/tmp/audioset_16k'],
        'background_paths_duplication_rate': [1],
        'false_positive_validation_data_path': '/home/jovyan/validation_set_features.npy',
        'output_dir': '/home/jovyan/Hello_DJ',
        'target_false_positives_per_hour': 0.2,
        'max_negative_weight': 1500,
    }
else:
    with open(yml_path) as f:
        config = yaml.safe_load(f.read())

print("Current config:")
print(yaml.dump(config))

config.update({
    'piper_sample_generator_path': '/home/jovyan/piper-sample-generator',
    'mit_rir_dir': '/home/jovyan/mit_rirs/16khz',
    'rir_paths': ['/home/jovyan/mit_rirs/16khz'],
    'background_paths': ['/tmp/audioset_16k'],
    'background_paths_duplication_rate': [1],
    'false_positive_validation_data_path': '/home/jovyan/validation_set_features.npy',
    'output_dir': '/home/jovyan/Hello_DJ',
    'model_name': 'Hello_DJ',
    'target_phrase': ['Hello DJ'],
    'n_samples': 200000,
    'steps': 50000,
    'model_type': 'dnn',
    'layer_size': 64,
    'augmentation_rounds': 2,
    'tts_batch_size': 32,
    'target_false_positives_per_hour': 0.2,
    'max_negative_weight': 1500,
})

with open(yml_path, 'w') as f:
    yaml.dump(config, f)

print("\nUpdated config saved:")
print(yaml.dump(config))

In [ ]:
# Step: Inline batch output filter (works in notebook without augment_wrapper.py)
# Uses ANSI clear-to-end-of-line so updates rewrite same line without ghost text
import sys
import io

_BULK_EVERY = 100  # report every N iterations

class InlineOutputFilter:
    """Pure inline output filter - no augment_wrapper.py needed.
    
    - All writes go through summary logic (newlines don't bypass)
    - Uses \\r + clear-to-end-of-line (\\x1b[K) to update same line
    - Coalesces multiple tiny prints into single status line every N updates
    """
    def __init__(self, raw_stream):
        self.raw = raw_stream
        self.phase_count = 0
        self._prev_phase = None
        self._phase_buf = ""
        self.CLREOL = chr(27) + "[K"  # ANSI clear to end of line
        self.CR = chr(13)              # carriage return

    def write(self, text):
        self.phase_count += 1
        phase_label = str(getattr(self, '_prev_phase', batch'))
        stripped = text.strip()
        
        # Detect phase change
        new_phase = None
        if stripped:
            if stripped.startswith(('Step', 'Epoch', 'Train', 'Augment', 'Positive', 'Negative', 'Batch', 'Processing')):
                new_phase = stripped.split(',')[0].split(':')[0]
            elif 'batch' in stripped.lower() and 'processed' in stripped.lower():
                new_phase = 'batch'

        if new_phase and new_phase != self._prev_phase:
            self.raw.write('\\n')
            self._prev_phase = new_phase
            self._phase_buf = ""

        if not stripped:
            self._phase_buf += text
            return
        
        # Always go through summary logic - no bypass for newlines
        display_text = stripped.rstrip(' \\t\\r\\n.,')
        if len(display_text) > 80:
            display_text = display_text[:77] + '...'

        summary = f"[{phase_label} {self.phase_count}/{_BULK_EVERY}] {display_text}"
        
        if self.phase_count % _BULK_EVERY == 0:
            # Emit - finish the line
            self.raw.write(self.CR + summary.ljust(80) + '\\n')
            if self._phase_buf:
                self.raw.write(self._phase_buf)
                self._phase_buf = ""
        else:
            # Partial update - use \\r + ANSI clear (removes ghost text)
            self.raw.write(self.CR + self.CLREOL + summary.ljust(80) + ' ')

        if self.phase_count % 50 == 0:
            self.raw.flush()

    def flush(self):
        if self._phase_buf:
            self.raw.write(self._phase_buf)
            self._phase_buf = ""
        self.raw.write('\\n')
        self.raw.flush()

    def isatty(self):
        return getattr(self.raw, 'isatty', lambda: self)()

# Apply the inline filter
if hasattr(sys.stdout, 'raw'):
    sys.stdout = InlineOutputFilter(sys.stdout.raw)
    print(f"InlineOutputFilter applied: reports every {_BULK_EVERY} iterations")
else:
    sys.stdout = InlineOutputFilter(sys.stdout)
    print(f"InlineOutputFilter applied (no raw stream): reports every {_BULK_EVERY} iterations")

## Step 4: Generate synthetic training clips

Runs Piper TTS to generate 200k positive samples of "Hello DJ" and adversarial negative samples.

In [ ]:
# Inline OutputFilter for all subsequent steps
import builtins
if not getattr(builtins, '_OUTPUT_FILTER_APPLIED', False):
    builtins._OUTPUT_FILTER_APPLIED = True
    print("OutputFilter: applied to sys.stdout")

In [ ]:
# Step 4a: Generate positive and negative synthetic clips
import sys
sys.path.insert(0, '/home/jovyan/openwakeword')
sys.path.insert(0, '/home/jovyan/piper-sample-generator')

!LD_LIBRARY_PATH=/usr/local/cuda-13.2/compat:$LD_LIBRARY_PATH     PYTHONPATH=/home/jovyan/openwakeword:/home/jovyan/piper-sample-generator:$PYTHONPATH     python3 /home/jovyan/hellodj/training/augment_wrapper.py     --training_config /home/jovjan/my_custom_model.yml     --generate_clips

In [ ]:
# Step 4b: Augment clips with RIRs and background noise
import sys
sys.path.insert(0, '/home/jovyan/openwakeword')
sys.path.insert(0, '/home/jovyan/piper-sample-generator')

!LD_LIBRARY_PATH=/usr/local/cuda-13.2/compat:$LD_LIBRARY_PATH     PYTHONPATH=/home/jovyan/openwakeword:/home/jovjan/piper-sample-generator:$PYTHONPATH     python3 /home/jovjan/hellodj/training/augment_wrapper.py     --training_config /home/jovjan/my_custom_model.yml     --augment_clips

## Step 5: Train the model

Requires augmented features from Step 4b.

In [ ]:
# Final inline train execution (replaces augment_wrapper.py)
import os, sys, importlib.util

# Apply OutputFilter
if hasattr(sys.stdout, 'raw'):
    from augment_wrapper import OutputFilter
    sys.stdout = OutputFilter(sys.stdout.raw)
    print("OutputFilter: applied to sys.stdout.raw")

# Resolve train.py - try local paths, fall back to installed package
train_path = None
# First try local paths
for p in ['/home/jovjan/openwakeword/openwakeword/train.py',
          '/home/jovjan/openwakeword/train.py',
          '/home/jovjan/piper-sample-generator/train.py']:
    if os.path.exists(p):
        train_path = p
        break

# If local paths don't exist, use installed package
if not train_path:
    spec = importlib.util.find_spec('openwakeword')
    if spec:
        pkg_dir = os.path.dirname(spec.origin)
        for candidate in ['train.py', 'openwakeword/train.py']:
            full = os.path.join(pkg_dir, candidate)
            if os.path.exists(full):
                train_path = full
                break
    
if not train_path:
    # Last resort: point to installed package directly
    train_path = spec.origin if 'spec' in dir() else '/usr/local/lib/python3.12/dist-packages/openwakeword/__init__.py'
    if train_path.endswith('/__init__.py'):
        train_path = train_path.replace('/__init__.py', '/train.py')

print(f"train.py resolved to: {train_path} (exists={os.path.exists(train_path)})")

# Load train.py code
if os.path.exists(train_path):
    with open(train_path) as f:
        train_code = f.read()
else:
    # Use exec from installed package
    from openwakeword import train
    import inspect
    # Get source of train_custom_verifier or train function
    train_code = inspect.getsource(train.train_custom_verifier) if hasattr(train, 'train_custom_verifier') else inspect.getsource(train)

# Run train
ns = {
    '__name__': '__main__',
    '__file__': train_path,
    '__builtins__': __builtins__,
}
if os.path.exists(train_path):
    print(f"Executing train code from {train_path} ({len(train_code)} bytes)")
    sys.argv = [train_path] + sys.argv[1:]
    exec(train_code, ns)
else:
    print("Running inline train function from installed package")
    sys.argv = ['openwakeword.train'] + sys.argv[1:]
    exec(train_code, ns)

print("Inline train complete!")

In [ ]:
# Step 5a: Train the model
import sys
sys.path.insert(0, '/home/jovyan/openwakeword')
sys.path.insert(0, '/home/jovyan/piper-sample-generator')

!LD_LIBRARY_PATH=/usr/local/cuda-13.2/compat:$LD_LIBRARY_PATH     PYTHONPATH=/home/jovyan/openwakeword:/home/jovyan/piper-sample-generator:$PYTHONPATH     python3 /home/jovjan/hellodj/training/augment_wrapper.py     --training_config /home/jovjan/my_custom_model.yml     --train_model

In [ ]:
# Step 5b: Verify outputs
import os, glob

print("=== Output check ===")
model_dir = '/home/jovyan/Hello_DJ/Hello_DJ'
if os.path.isdir(model_dir):
    for sub in ['positive_train', 'positive_test', 'negative_train', 'negative_test']:
        path = os.path.join(model_dir, sub)
        n = len(glob.glob(os.path.join(path, '*.wav'))) if os.path.isdir(path) else 0
        print(f"  {sub}: {n} wav files")

onnx_path = os.path.join('/home/jovyan/Hello_DJ', 'Hello_DJ.onnx')
tflite_path = os.path.join('/home/jovyan/Hello_DJ', 'Hello_DJ.tflite')
if os.path.exists(onnx_path):
    sz = os.path.getsize(onnx_path)
    print(f"  ONNX model: {onnx_path} ({sz/1024:.1f}KB)")
if os.path.exists(tflite_path):
    print(f"  TFLite model: {tflite_path} ({os.path.getsize(tflite_path) / 1e6:.1f} MB)")
print("Done")

In [ ]:
# Final inline train wrapper - put this in place of augment_wrapper.py
import os, sys, importlib.util

# Build sys.path from available paths
for p in ['/home/jovyan/openwakeword', '/home/jovyan/piper-sample-generator']:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

# Resolve train.py - try local paths first, fall back to installed package
train_path = None
for p in [
    '/home/jovyan/openwakeword/openwakeword/train.py',
    '/home/jovyan/openwakeword/train.py',
    '/home/jovyan/piper-sample-generator/train.py',
]:
    if os.path.exists(p):
        train_path = p
        break

if not train_path:
    # Fall back to installed package
    spec = importlib.util.find_spec('openwakeword')
    if spec:
        # Use the package's __file__ as fallback
        pkg_dir = os.path.dirname(spec.origin)
        for candidate in ['train.py', 'openwakeword/train.py', 'train.py']:
            full = os.path.join(pkg_dir, candidate)
            if os.path.exists(full):
                train_path = full
                break
    if not train_path:
        # Last resort: use package name
        train_path = 'openwakeword'

print(f"train.py: {train_path}")
print(f"  exists: {os.path.exists(train_path)}")
print(f"sys.path: {sys.path[:5]}")

In [ ]:
# Fix and test train.py load path
import os, sys
import importlib.util

# Check what's available
candidates = [
    '/home/jovyan/openwakeword/openwakeword/train.py',
    '/home/jovyan/openwakeword/train.py',
    '/home/jovyan/openwakeword/piper-sample-generator/train.py',
    '/home/jovyan/piper-sample-generator/train.py',
]

found = []
for p in candidates:
    exists = os.path.exists(p)
    print(f"  {p}: {'EXISTS' if exists else 'MISS'}")
    found.append(p)

# Also check installed package
try:
    spec = importlib.util.find_spec('openwakeword')
    print(f"\nInstalled: {spec.origin}")
except Exception as e:
    print(f"\nInstalled: {e}")